
# Flight Delay & Cancellation Dashboard (RAPIDS + cuDF + Plotly + Dash)

This notebook builds a GPU-accelerated dashboard in **Google Colab** using:
- **NVIDIA RAPIDS / cuDF** for loading and aggregating a large flight dataset
- **Plotly** for charts
- **Dash / JupyterDash** for the interactive dashboard UI

It is designed for your **full 3M-row CSV**, not just the attached sample file.

## What this notebook does
- Loads the full CSV with **cuDF**
- Creates time-based and delay/cancellation features
- Builds a dark themed dashboard inspired by your screenshot
- Adds filters for **Year** and **Airline**
- Shows KPI cards plus 6 charts:
  1. Average departure delay by month
  2. Departure delay distribution
  3. Cancellation reasons
  4. Average departure delay heatmap (day x month)
  5. Top airlines by average departure delay
  6. Top airlines by cancellation rate

## Before you run
In Colab, go to:
**Runtime -> Change runtime type -> Hardware accelerator -> GPU**

A **T4**, **L4**, or better GPU is recommended.


In [ ]:

# Step 1: Install RAPIDS in Colab
# This can take a few minutes.

!nvidia-smi
!git clone https://github.com/rapidsai-community/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py


In [ ]:

# Step 2: Install dashboard dependencies
!pip -q install dash plotly jupyter-dash


In [ ]:

# Step 3: Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cudf
import cupy as cp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from dash import dcc, html, Input, Output
from jupyter_dash import JupyterDash
from google.colab import drive

print('cuDF version:', cudf.__version__)



## Point the notebook to your full CSV

Upload the full file to **Google Drive** first, then update `CSV_PATH` below.

Example:
`/content/drive/MyDrive/DATA230/cleaned_flights_full.csv`


In [ ]:

# Step 4: Mount Google Drive and set your file path

drive.mount('/content/drive')

CSV_PATH = '/content/drive/MyDrive/cleaned_flights_full.csv'  # <-- change this

# For quick testing only, you can temporarily point to the sample file if you uploaded it into Colab.
# CSV_PATH = '/content/cleaned_flights_sample_1000_rows.csv'


In [ ]:

# Step 5: Load the CSV with cuDF
# These dtypes are tuned to reduce memory pressure for a large file.

dtype_map = {
    'AIRLINE': 'str',
    'AIRLINE_CODE': 'str',
    'ORIGIN': 'str',
    'DEST': 'str',
    'ORIGIN_CITY_NAME': 'str',
    'ORIGIN_STATE': 'str',
    'DEST_CITY_NAME': 'str',
    'DEST_STATE': 'str',
    'CANCELLATION_CODE': 'str',
    'FL_NUMBER': 'int32',
    'DOT_CODE': 'int32',
    'CRS_DEP_TIME': 'float32',
    'DEP_TIME': 'float32',
    'DEP_DELAY': 'float32',
    'TAXI_OUT': 'float32',
    'WHEELS_OFF': 'float32',
    'WHEELS_ON': 'float32',
    'TAXI_IN': 'float32',
    'CRS_ARR_TIME': 'float32',
    'ARR_TIME': 'float32',
    'ARR_DELAY': 'float32',
    'CANCELLED': 'int8',
    'DIVERTED': 'int8',
    'CRS_ELAPSED_TIME': 'float32',
    'ELAPSED_TIME': 'float32',
    'AIR_TIME': 'float32',
    'DISTANCE': 'float32',
    'DELAY_DUE_CARRIER': 'float32',
    'DELAY_DUE_WEATHER': 'float32',
    'DELAY_DUE_NAS': 'float32',
    'DELAY_DUE_SECURITY': 'float32',
    'DELAY_DUE_LATE_AIRCRAFT': 'float32',
    'CRS_DEP_TIME_MIN': 'float32',
    'DEP_TIME_MIN': 'float32',
    'WHEELS_OFF_MIN': 'float32',
    'WHEELS_ON_MIN': 'float32',
    'CRS_ARR_TIME_MIN': 'float32',
    'ARR_TIME_MIN': 'float32',
    'SCHED_BLOCK_MINS': 'float32',
    'ACTUAL_BLOCK_MINS': 'float32',
    'DEP_DELAY_CLEAN': 'float32',
    'ARR_DELAY_CLEAN': 'float32',
    'TAXI_OUT_CLEAN': 'float32',
    'TAXI_IN_CLEAN': 'float32',
    'AIR_TIME_CLEAN': 'float32',
    'ELAPSED_TIME_CLEAN': 'float32'
}

date_cols = ['FL_DATE', 'CRS_DEP_DT', 'CRS_ARR_DT', 'DEP_DT', 'ARR_DT']

print('Loading CSV from:', CSV_PATH)
df = cudf.read_csv(CSV_PATH, dtype=dtype_map)

for col in date_cols:
    if col in df.columns:
        df[col] = cudf.to_datetime(df[col], errors='coerce')

print('Rows, columns:', df.shape)
print(df.head())


In [ ]:

# Step 6: Feature engineering for the dashboard

month_map = {
    1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
    7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'
}

cancel_map = {
    'A': 'Carrier',
    'B': 'Weather',
    'C': 'NAS',
    'D': 'Security',
    'NONE': 'Not Cancelled'
}

# Use cleaned delay column when available, otherwise DEP_DELAY
base_delay_col = 'DEP_DELAY_CLEAN' if 'DEP_DELAY_CLEAN' in df.columns else 'DEP_DELAY'
arr_delay_col = 'ARR_DELAY_CLEAN' if 'ARR_DELAY_CLEAN' in df.columns else 'ARR_DELAY'

df = df.dropna(subset=['FL_DATE'])

df['YEAR'] = df['FL_DATE'].dt.year.astype('int16')
df['MONTH_NUM'] = df['FL_DATE'].dt.month.astype('int8')
df['DAY_NUM'] = df['FL_DATE'].dt.weekday.astype('int8')

df['MONTH'] = df['MONTH_NUM'].map(month_map)
df['DAY_NAME'] = df['DAY_NUM'].map({0:'Monday',1:'Tuesday',2:'Wednesday',3:'Thursday',4:'Friday',5:'Saturday',6:'Sunday'})

# Fill nulls in key columns used by the dashboard
for col in [base_delay_col, arr_delay_col, 'CANCELLED', 'AIRLINE', 'CANCELLATION_CODE']:
    if col in df.columns:
        if df[col].dtype == 'object' or str(df[col].dtype) == 'str':
            df[col] = df[col].fillna('Unknown')
        else:
            df[col] = df[col].fillna(0)

# Cancellation reason labels
if 'CANCELLATION_CODE' in df.columns:
    df['CANCEL_REASON'] = df['CANCELLATION_CODE'].astype('str').str.upper().map(cancel_map).fillna('Other')
else:
    df['CANCEL_REASON'] = 'Unknown'

# Delay buckets
condlist = [
    df[base_delay_col] <= 0,
    (df[base_delay_col] > 0) & (df[base_delay_col] <= 15),
    (df[base_delay_col] > 15) & (df[base_delay_col] <= 45),
    (df[base_delay_col] > 45) & (df[base_delay_col] <= 120),
    df[base_delay_col] > 120
]
choicelist = ['On time / early', '0-15 min', '15-45 min', '45-120 min', '>120 min']
df['DEP_DELAY_BUCKET'] = cudf.Series(cp.select(condlist, choicelist, default='Unknown'))

# Optional: cast string columns to category-like storage if helpful
# df['AIRLINE'] = df['AIRLINE'].astype('category')

print(df[[ 'FL_DATE', 'YEAR', 'MONTH', 'DAY_NAME', base_delay_col, 'DEP_DELAY_BUCKET', 'CANCEL_REASON']].head())


In [ ]:

# Step 7: Quick health check
print('Total rows:', len(df))
print('Years:', sorted(df['YEAR'].dropna().unique().to_pandas().tolist()))
print('Airlines:', len(df['AIRLINE'].dropna().unique()))
print('GPU memory friendly tip: keep only needed columns if your runtime is tight.')


In [ ]:

# Step 8: Dashboard helper functions

MONTH_ORDER = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
DAY_ORDER = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
BUCKET_ORDER = ['On time / early', '0-15 min', '15-45 min', '45-120 min', '>120 min']


def filter_df(gdf, year_value, airline_value):
    out = gdf
    if year_value != 'All':
        out = out[out['YEAR'] == int(year_value)]
    if airline_value != 'All':
        out = out[out['AIRLINE'] == airline_value]
    return out


def fmt_int(x):
    return f"{int(x):,}"


def fmt_pct(x):
    return f"{x:.2f}%"


def fmt_min(x):
    return f"{x:.1f} min"


def kpi_metrics(gdf):
    total_flights = int(len(gdf))
    delayed_15 = float(((gdf[base_delay_col] > 15).sum() / max(total_flights, 1)) * 100)
    avg_delay = float(gdf[base_delay_col].mean()) if total_flights else 0.0
    cancel_rate = float((gdf['CANCELLED'].sum() / max(total_flights, 1)) * 100)
    return total_flights, delayed_15, avg_delay, cancel_rate


def monthly_delay_fig(gdf):
    grp = gdf.groupby(['YEAR', 'MONTH_NUM', 'MONTH'])[base_delay_col].mean().reset_index()
    pdf = grp.to_pandas().sort_values(['YEAR', 'MONTH_NUM'])
    fig = px.line(
        pdf,
        x='MONTH', y=base_delay_col, color='YEAR', markers=True,
        category_orders={'MONTH': MONTH_ORDER},
        title='Avg departure delay by month (min)'
    )
    fig.update_layout(template='plotly_dark', height=360, margin=dict(l=20, r=20, t=50, b=20), legend_title_text='Year')
    fig.update_xaxes(title=None)
    fig.update_yaxes(title=None)
    return fig


def delay_distribution_fig(gdf):
    grp = gdf.groupby('DEP_DELAY_BUCKET').size().reset_index(name='COUNT')
    pdf = grp.to_pandas()
    total = max(pdf['COUNT'].sum(), 1)
    pdf['PCT'] = (pdf['COUNT'] / total) * 100
    pdf['DEP_DELAY_BUCKET'] = pd.Categorical(pdf['DEP_DELAY_BUCKET'], categories=BUCKET_ORDER, ordered=True)
    pdf = pdf.sort_values('DEP_DELAY_BUCKET')
    fig = px.bar(pdf, x='DEP_DELAY_BUCKET', y='COUNT', text=pdf['PCT'].round(1).astype(str) + '%',
                 title='Departure delay distribution')
    fig.update_layout(template='plotly_dark', height=360, margin=dict(l=20, r=20, t=50, b=20))
    fig.update_xaxes(title=None)
    fig.update_yaxes(title=None)
    return fig


def cancellation_reasons_fig(gdf):
    cancelled = gdf[gdf['CANCELLED'] == 1]
    if len(cancelled) == 0:
        fig = go.Figure()
        fig.update_layout(template='plotly_dark', title='Cancellation reasons', height=320)
        fig.add_annotation(text='No cancelled flights in current selection', showarrow=False)
        return fig
    grp = cancelled.groupby('CANCEL_REASON').size().reset_index(name='COUNT')
    pdf = grp.to_pandas().sort_values('COUNT', ascending=False)
    fig = px.pie(pdf, values='COUNT', names='CANCEL_REASON', hole=0.58, title='Cancellation reasons')
    fig.update_layout(template='plotly_dark', height=320, margin=dict(l=20, r=20, t=50, b=20))
    return fig


def delay_heatmap_fig(gdf):
    grp = gdf.groupby(['DAY_NAME', 'MONTH'])[base_delay_col].mean().reset_index()
    pdf = grp.to_pandas()
    if pdf.empty:
        fig = go.Figure()
        fig.update_layout(template='plotly_dark', title='Avg departure delay (min) — day x month', height=320)
        return fig
    piv = pdf.pivot(index='DAY_NAME', columns='MONTH', values=base_delay_col)
    piv = piv.reindex(index=DAY_ORDER, columns=MONTH_ORDER)
    fig = px.imshow(
        piv,
        text_auto='.1f',
        aspect='auto',
        title='Avg departure delay (min) — day x month',
        color_continuous_scale='RdBu_r'
    )
    fig.update_layout(template='plotly_dark', height=320, margin=dict(l=20, r=20, t=50, b=20), coloraxis_colorbar_title='Min')
    fig.update_xaxes(title=None)
    fig.update_yaxes(title=None)
    return fig


def top_airlines_delay_fig(gdf, top_n=10):
    grp = gdf.groupby('AIRLINE')[base_delay_col].mean().reset_index()
    pdf = grp.to_pandas().sort_values(base_delay_col, ascending=False).head(top_n)
    pdf = pdf.sort_values(base_delay_col, ascending=True)
    fig = px.bar(pdf, x=base_delay_col, y='AIRLINE', orientation='h', title='Top airlines by avg departure delay (min)', text=base_delay_col.round(1))
    fig.update_layout(template='plotly_dark', height=340, margin=dict(l=20, r=20, t=50, b=20))
    fig.update_xaxes(title=None)
    fig.update_yaxes(title=None)
    return fig


def top_airlines_cancel_fig(gdf, top_n=10):
    grp = gdf.groupby('AIRLINE').agg({'CANCELLED': 'mean'}).reset_index()
    grp['CANCEL_RATE_PCT'] = grp['CANCELLED'] * 100
    pdf = grp[['AIRLINE', 'CANCEL_RATE_PCT']].to_pandas().sort_values('CANCEL_RATE_PCT', ascending=False).head(top_n)
    pdf = pdf.sort_values('CANCEL_RATE_PCT', ascending=True)
    fig = px.bar(pdf, x='CANCEL_RATE_PCT', y='AIRLINE', orientation='h', title='Top airlines by cancellation rate (%)', text=pdf['CANCEL_RATE_PCT'].round(2))
    fig.update_layout(template='plotly_dark', height=340, margin=dict(l=20, r=20, t=50, b=20))
    fig.update_xaxes(title=None)
    fig.update_yaxes(title=None)
    return fig


In [ ]:

# Step 9: Build dashboard options

year_values = sorted(df['YEAR'].dropna().unique().to_pandas().tolist())
airline_values = sorted(df['AIRLINE'].dropna().unique().to_pandas().tolist())

year_options = [{'label': 'All years', 'value': 'All'}] + [{'label': str(y), 'value': str(y)} for y in year_values]
airline_options = [{'label': 'All airlines', 'value': 'All'}] + [{'label': a, 'value': a} for a in airline_values]

subtitle = f"{min(year_values)}-{max(year_values)} · US domestic flights · {len(df):,} rows loaded"
print(subtitle)


In [ ]:

# Step 10: Create the Plotly Dash app

app = JupyterDash(__name__)
server = app.server

CARD_STYLE = {
    'backgroundColor': '#141c2b',
    'borderRadius': '12px',
    'padding': '14px 18px',
    'boxShadow': '0 0 0 1px rgba(255,255,255,0.04) inset'
}

SECTION_STYLE = {
    'backgroundColor': '#141c2b',
    'borderRadius': '12px',
    'padding': '10px 10px 0px 10px',
    'boxShadow': '0 0 0 1px rgba(255,255,255,0.04) inset'
}

app.layout = html.Div([
    html.Div([
        html.H2('✈ Flight Delay & Cancellation Overview', style={'marginBottom': '4px', 'color': 'white'}),
        html.Div(subtitle, style={'color': '#9aa4b2', 'fontSize': '13px', 'marginBottom': '18px'}),

        html.Div([
            html.Div([
                html.Div('Year', style={'fontSize': '12px', 'color': '#9aa4b2', 'marginBottom': '6px'}),
                dcc.Dropdown(id='year-filter', options=year_options, value='All', clearable=False)
            ], style={'width': '220px'}),

            html.Div([
                html.Div('Airline', style={'fontSize': '12px', 'color': '#9aa4b2', 'marginBottom': '6px'}),
                dcc.Dropdown(id='airline-filter', options=airline_options, value='All', clearable=False)
            ], style={'width': '320px'})
        ], style={'display': 'flex', 'gap': '16px', 'marginBottom': '16px'}),

        html.Div([
            html.Div([html.Div('TOTAL FLIGHTS', style={'fontSize': '11px', 'color': '#9aa4b2'}), html.H3(id='kpi-total', style={'margin': '6px 0 0 0'})], style=CARD_STYLE),
            html.Div([html.Div('DELAYED (>15 MIN)', style={'fontSize': '11px', 'color': '#9aa4b2'}), html.H3(id='kpi-delayed', style={'margin': '6px 0 0 0', 'color': '#66a3ff'})], style=CARD_STYLE),
            html.Div([html.Div('AVG DELAY', style={'fontSize': '11px', 'color': '#9aa4b2'}), html.H3(id='kpi-avg-delay', style={'margin': '6px 0 0 0', 'color': '#f0c36d'})], style=CARD_STYLE),
            html.Div([html.Div('CANCELLATION RATE', style={'fontSize': '11px', 'color': '#9aa4b2'}), html.H3(id='kpi-cancel', style={'margin': '6px 0 0 0', 'color': '#ff6b6b'})], style=CARD_STYLE),
        ], style={'display': 'grid', 'gridTemplateColumns': 'repeat(4, 1fr)', 'gap': '12px', 'marginBottom': '12px'}),

        html.Div([
            html.Div(dcc.Graph(id='fig-monthly'), style=SECTION_STYLE),
            html.Div(dcc.Graph(id='fig-distribution'), style=SECTION_STYLE),
        ], style={'display': 'grid', 'gridTemplateColumns': '2fr 1fr', 'gap': '12px', 'marginBottom': '12px'}),

        html.Div([
            html.Div(dcc.Graph(id='fig-cancel-reasons'), style=SECTION_STYLE),
            html.Div(dcc.Graph(id='fig-heatmap'), style=SECTION_STYLE),
        ], style={'display': 'grid', 'gridTemplateColumns': '1fr 2fr', 'gap': '12px', 'marginBottom': '12px'}),

        html.Div([
            html.Div(dcc.Graph(id='fig-top-delay'), style=SECTION_STYLE),
            html.Div(dcc.Graph(id='fig-top-cancel'), style=SECTION_STYLE),
        ], style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '12px'}),
    ], style={'maxWidth': '1500px', 'margin': '0 auto', 'padding': '14px'})
], style={'backgroundColor': '#0b1020', 'minHeight': '100vh', 'fontFamily': 'Arial, sans-serif'})


@app.callback(
    Output('kpi-total', 'children'),
    Output('kpi-delayed', 'children'),
    Output('kpi-avg-delay', 'children'),
    Output('kpi-cancel', 'children'),
    Output('fig-monthly', 'figure'),
    Output('fig-distribution', 'figure'),
    Output('fig-cancel-reasons', 'figure'),
    Output('fig-heatmap', 'figure'),
    Output('fig-top-delay', 'figure'),
    Output('fig-top-cancel', 'figure'),
    Input('year-filter', 'value'),
    Input('airline-filter', 'value')
)
def update_dashboard(year_value, airline_value):
    dff = filter_df(df, year_value, airline_value)
    total_flights, delayed_15, avg_delay, cancel_rate = kpi_metrics(dff)

    return (
        fmt_int(total_flights),
        fmt_pct(delayed_15),
        fmt_min(avg_delay),
        fmt_pct(cancel_rate),
        monthly_delay_fig(dff),
        delay_distribution_fig(dff),
        cancellation_reasons_fig(dff),
        delay_heatmap_fig(dff),
        top_airlines_delay_fig(dff),
        top_airlines_cancel_fig(dff)
    )


In [ ]:

# Step 11: Run the dashboard in Colab
# After execution, click the generated link or use the inline frame.

app.run_server(mode='inline', debug=False, port=8050)



## Notes for a 3M-row file

### 1) Why cuDF helps
For this project, the heavy work is:
- filtering by year and airline
- grouping by month / day / airline
- computing means, counts, and rates

Those operations are much faster on the GPU than doing them on a regular pandas DataFrame in Colab.

### 2) If memory gets tight
Keep only the columns used by the dashboard. Add a cell like this **right after loading**:

```python
needed_cols = [
    'FL_DATE', 'AIRLINE', 'CANCELLED', 'CANCELLATION_CODE',
    'DEP_DELAY', 'DEP_DELAY_CLEAN'
]
needed_cols = [c for c in needed_cols if c in df.columns]
df = df[needed_cols].copy()
```

### 3) If the CSV is very large to reload each session
Convert it once to Parquet and reuse it:

```python
PARQUET_PATH = '/content/drive/MyDrive/cleaned_flights_full.parquet'
df.to_parquet(PARQUET_PATH, index=False)
```

Then later load with:

```python
df = cudf.read_parquet(PARQUET_PATH)
```

That is usually faster than reading CSV every time.

### 4) If the dashboard feels slow
Create pre-aggregated summary tables by:
- year x month
- year x airline
- year x airline x delay bucket
- year x airline x cancellation reason

Then the callback only reads the already grouped results.



## What to submit / show in class or in your report
- A screenshot of the dashboard running in Colab
- A short note that the dashboard uses **RAPIDS cuDF** for GPU data processing
- A note that the dashboard is built for the **full 3M-row dataset**, not only the sample CSV
- A short explanation of each KPI and chart
